# Create the DistilBERT LibTorch Artifact

Run this notebook in a Development workspace based on `nvcr.io/nvidia/pytorch:26.06-py3`. The PyTorch image already provides a GPU-capable `torch` build for Blackwell GPUs such as RTX 5080. Do not install `torch` in this notebook.

## 1. Bootstrap

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time


# The workspace container runs as uid 10001, which may not exist in /etc/passwd.
# Set these before importing torch/transformers so PyTorch cache setup is stable.
os.environ.setdefault("USER", "workspace")
os.environ.setdefault("LOGNAME", "workspace")
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", "/tmp/torchinductor-workspace")

MODEL_ID = "distilbert-base-uncased-finetuned-sst-2-english"
MAX_LENGTH = 32
OUTPUT_PATH = Path("distilbert_sentiment/1/model.pt")

print("Python:", sys.executable)
print("Model:", MODEL_ID)
print("Output:", OUTPUT_PATH)

## 2. Install Transformers if needed

In [ ]:
if importlib.util.find_spec("transformers") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--user",
        "transformers==4.53.0",
    ])
    print("Installed transformers. Restart the kernel, then run the notebook from the top.")
else:
    print("transformers is already installed")

If the previous cell installed `transformers`, restart the kernel now and run from the top. Do not install `torch`.

## 3. Check PyTorch, Transformers, and GPU

In [ ]:
import torch
import transformers


print("torch:", torch.__version__)
print("torch file:", torch.__file__)
print("torch CUDA:", torch.version.cuda)
print("transformers:", transformers.__version__)
print("transformers file:", transformers.__file__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit("CUDA is required to export this GPU LibTorch example.")

print("CUDA device:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))
print("CUDA arch list:", torch.cuda.get_arch_list())

if "sm_120" not in torch.cuda.get_arch_list():
    raise SystemExit("This PyTorch build does not support sm_120. Use nvcr.io/nvidia/pytorch:26.06-py3 or newer.")

if "/workspace/.local" in torch.__file__:
    raise SystemExit("torch is loaded from /workspace/.local. Remove user-installed torch and restart the kernel.")

## 4. Define helpers

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer


def load_from_huggingface(loader, model_id: str, label: str, attempts: int = 3):
    last_error = None
    for attempt in range(1, attempts + 1):
        print(f"Loading {label} from {model_id} (attempt {attempt}/{attempts})...")
        try:
            result = loader.from_pretrained(model_id)
            print(f"Loaded {label}")
            return result
        except (OSError, RuntimeError) as exc:
            last_error = exc
            if attempt == attempts:
                break
            wait_seconds = attempt * 2
            print(f"Could not load {label}, retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)
    raise RuntimeError(
        f"Could not load {label} from {model_id}. Check workspace DNS/internet access "
        "to huggingface.co, or pre-populate the HuggingFace cache before running this notebook."
    ) from last_error

In [ ]:
class SentimentWrapper(torch.nn.Module):
    def __init__(self, model: torch.nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return output.logits

## 5. Load the HuggingFace model

In [ ]:
config = load_from_huggingface(AutoConfig, MODEL_ID, "model config")
print("Model type:", config.model_type)

In [ ]:
tokenizer = load_from_huggingface(AutoTokenizer, MODEL_ID, "tokenizer")

In [ ]:
base_model = load_from_huggingface(
    AutoModelForSequenceClassification,
    MODEL_ID,
    "sequence classification model",
)
base_model.eval()
print("Model loaded on CPU")

## 6. Move to CUDA

In [ ]:
device = torch.device("cuda")
base_model = base_model.to(device)
base_model.eval()
print("Model moved to CUDA")

## 7. Trace TorchScript

In [ ]:
encoded = tokenizer(
    "This product is genuinely useful and easy to recommend.",
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

example_inputs = (
    encoded["input_ids"].to(device=device, dtype=torch.long),
    encoded["attention_mask"].to(device=device, dtype=torch.long),
)

print("input_ids shape:", tuple(example_inputs[0].shape))
print("attention_mask shape:", tuple(example_inputs[1].shape))

In [ ]:
model = SentimentWrapper(base_model).to(device)
model.eval()

with torch.inference_mode():
    logits = model(*example_inputs)
    print("Smoke-test logits shape:", tuple(logits.shape))
    traced_model = torch.jit.trace(model, example_inputs, strict=False)

print("TorchScript trace created")

## 8. Save the Triton artifact

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
traced_model.save(str(OUTPUT_PATH))
print(f"Saved {OUTPUT_PATH}")